# Очистка и нормализация текста

Прежде чем построить эффективную RAG-систему, нужно решить базовую проблему: сырые документы из реального мира редко приходят в готовом для использования виде. PDF-файлы содержат колонтитулы и номера страниц, HTML-страницы — навигационные меню и рекламу, сканированные документы — артефакты распознавания текста.

Хорошая очистка повышает качество последующего поиска и генерации ответов, потому что модель получает более релевантный контент. Документы из реального мира часто содержат артефакты: номера страниц, колонтитулы, повторяющиеся служебные блоки, неравномерный регистр, опечатки OCR, фрагменты кода или разметки.

Всё это — шум, который снижает точность эмбеддингов и засоряет результаты поиска.

### Гибридный подход — текст + структура:

In [1]:
import pandas as pd


def extract_table_as_dataframe(table_text: str) -> pd.DataFrame:
    """Преобразуем таблицу в DataFrame."""
    lines = table_text.strip().split("\n")
    rows = []
    for line in lines:
        if "|" in line:
            row = [cell.strip() for cell in line.split("|") if cell.strip()]
        else:
            row = [cell.strip() for cell in line.split("\t") if cell.strip()]
        if row:
            rows.append(row)
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows[1:], columns=rows[0])
    return df


def table_as_string(df: pd.DataFrame) -> str:
    """Создаём описательный текст для каждой строки."""
    if df.empty:
        return ""
    descriptions = []
    for _, row in df.iterrows():
        desc = ". ".join([f"{col}: {row[col]}" for col in df.columns])
        descriptions.append(desc)
    return "\n".join(descriptions)


table_example = """
| Продукт | Цена | Количество | Статус |
| Ноутбук | 50000 | 5 | В наличии |
| Монитор | 15000 | 3 | Ограничено |
| Клавиатура | 5000 | 10 | В наличии |
"""
df = extract_table_as_dataframe(table_example)

print("📊 Структурированные данные:")
print(df.to_dict(orient="records"))
print("\n📝 Описательный текст:")
print(table_as_string(df))

📊 Структурированные данные:
[{'Продукт': 'Ноутбук', 'Цена': '50000', 'Количество': '5', 'Статус': 'В наличии'}, {'Продукт': 'Монитор', 'Цена': '15000', 'Количество': '3', 'Статус': 'Ограничено'}, {'Продукт': 'Клавиатура', 'Цена': '5000', 'Количество': '10', 'Статус': 'В наличии'}]

📝 Описательный текст:
Продукт: Ноутбук. Цена: 50000. Количество: 5. Статус: В наличии
Продукт: Монитор. Цена: 15000. Количество: 3. Статус: Ограничено
Продукт: Клавиатура. Цена: 5000. Количество: 10. Статус: В наличии


## Интеграция в пайплайн LangChain

In [ ]:
import bs4
from langchain_community.document_loaders import PyMuPDFLoader, WebBaseLoader
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langchain_text_splitters import TokenTextSplitter


# 1. СОЗДАНИЕ RUNNABLE ЗАГРУЗЧИКОВ
class LoaderRunnable(RunnableLambda):
    def __init__(self, loader):
        super().__init__(lambda _: list(loader.lazy_load()))
        self.loader = loader


load_pdf = LoaderRunnable(
    PyMuPDFLoader("docs/document.pdf"),
)
load_html = LoaderRunnable(
    WebBaseLoader(
        web_paths=("https://docs.langchain.com/oss/python/langchain/overview",),
        bs_kwargs={"parse_only": bs4.SoupStrainer(id="content")},
    )
)


# 2. СОЗДАНИЕ RUNNABLE ОБРАБОТЧИКОВ (Функции обработки тут - это просто заглушки)
def clean_pdf_text(text: str) -> str:
    return text


def clean_html_text(text: str) -> str:
    return text


def normalize_text(text: str) -> str:
    return text.lower()


def apply_func_to_all_docs(func):
    def process_docs(docs):
        for doc in docs:
            doc.page_content = func(doc.page_content)
        return docs

    return process_docs


clean_pdf = RunnableLambda(apply_func_to_all_docs(clean_pdf_text))
clean_html = RunnableLambda(apply_func_to_all_docs(clean_html_text))
normalize_all = RunnableLambda(apply_func_to_all_docs(normalize_text))


# 4. КОМПОЗИЦИЯ ЦЕПОЧКИ
chain = (
    RunnableParallel(pdf=load_pdf | clean_pdf, html=load_html | clean_html)
    | RunnableLambda(lambda x: x["pdf"] + x["html"])
    | normalize_all
)

# 5. ВЫЗОВ
result = chain.invoke(None)

# 6. Смотрим результат
for i, doc in enumerate(result):
    print(f"=== Content {i + 1} ===")
    print(doc.page_content[:50])
    print("Источник:", doc.metadata["source"], "\n")

USER_AGENT environment variable not set, consider setting it to identify your requests.


=== Content 1 ===
original paper
the catalyst selectivity index (csi
Источник: docs/document.pdf 

=== Content 2 ===
course, retain these deﬁnitions of the catalytic p
Источник: docs/document.pdf 

=== Content 3 ===
which is, of course, dictated by the intrinsic phy
Источник: docs/document.pdf 

=== Content 4 ===
2.2 goal and scope deﬁnition
astheuseofvariouscarb
Источник: docs/document.pdf 

=== Content 5 ===
displaced input factor. co-product credits are fur
Источник: docs/document.pdf 

=== Content 6 ===
3 the catalyst sensitivity index (csi): deﬁnition

Источник: docs/document.pdf 

=== Content 7 ===
improvements in catalyst efﬁciencies would not sig
Источник: docs/document.pdf 

=== Content 8 ===
coal-to-liquid process, as the h2/co ratio is rela
Источник: docs/document.pdf 

=== Content 9 ===
a further detailed breakdown of the complete wtt
l
Источник: docs/document.pdf 

=== Content 10 ===
process of the steam methane reforming (smr). ther
Источник: docs/document.pdf 

=== Conte

## Chunking

### Разбиение по символам

In [8]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import FileSystemBlobLoader
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import PyPDFParser

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)


loader = GenericLoader(
    blob_loader=FileSystemBlobLoader(path="docs", glob="*.pdf"),
    blob_parser=PyPDFParser(),
)
docs = loader.load()

chunks = text_splitter.split_documents(docs)

print(f"Было документов: {len(docs)}, стало фрагментов: {len(chunks)}")

Было документов: 14, стало фрагментов: 85


### Разбиение по токенам

In [ ]:
from langchain_text_splitters import TokenTextSplitter

text_splitter = TokenTextSplitter(
    encoding_name="cl100k_base", chunk_size=200, chunk_overlap=20
)
splitted_docs = text_splitter.split_documents(all_docs)

print(f"Было документов: {len(all_docs)}, стало фрагментов: {len(splitted_docs)}")

Было документов: 0, стало фрагментов: 0


### Семантическое разбиение текста


In [ ]:
from langchain_text_splitters import (
    HTMLHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

# Шаг 1: Разбиение по HTML-заголовкам
headers_to_split_on = [("h1", "Header 1"), ("h2", "Header 2"), ("h3", "Header 3")]
html_splitter = HTMLHeaderTextSplitter(headers_to_split_on)
html_header_splits = html_splitter.split_text(html_text)

# Шаг 2: Дополнительное разбиение по размеру
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(html_header_splits)

## Embeddings models

In [18]:
from langchain_huggingface import HuggingFaceEmbeddings

embed_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vector = embed_model.embed_query("Пример текста для эмбеддинга")
print(len(vector), vector[:5])

384 [-0.0026633096858859062, 0.0964525043964386, 0.05379258468747139, 0.027898533269762993, -0.07026731967926025]


In [ ]:
import os

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()

embed_model = OpenAIEmbeddings(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="text-embedding-ada-002",
)

vector = embed_model.embed_query("Пример текста для эмбеддинга")
print(len(vector), vector[:5])

1536 [-0.03590132296085358, 0.008473309688270092, -0.01868334226310253, -0.008466525934636593, 0.0009828428737819195]
